In [6]:
import os
import sqlite3
import pandas as pd

folder = './dataset.csv'
file_name = 'dataset.csv'
csv_path = os.path.join(folder, file_name)

if not os.path.isdir(folder):
    raise FileNotFoundError(f"No existe la carpeta: {folder}")

if not os.path.isfile(csv_path):
    raise FileNotFoundError(f"No existe el archivo CSV: {csv_path}")

try:
    df = pd.read_csv(csv_path)
    print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
    print("Columnas disponibles:", list(df.columns))

    conn = sqlite3.connect('spotify.db')
    cursor = conn.cursor()
    cursor.execute('PRAGMA foreign_keys = OFF;')

    # 1. albums
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS albums (
            track_id TEXT PRIMARY KEY,
            track_name TEXT NOT NULL,
            album_name TEXT,
            artists TEXT,
            popularity INTEGER
        )
    ''')

    df_albums = (
        df[['track_id', 'track_name', 'album_name', 'artists', 'popularity']]
        .drop_duplicates(subset=['track_id'])
        .fillna('')
    )

    lista_tuplas_albums = list(df_albums.itertuples(index=False, name=None))

    cursor.executemany('''
        INSERT OR REPLACE INTO albums (
            track_id, track_name, album_name, artists, popularity
        )
        VALUES (?, ?, ?, ?, ?)
    ''', lista_tuplas_albums)

    print(f"✅ Insertados {len(lista_tuplas_albums)} álbumes únicos")

    # 2. genre
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS genre (
            track_id TEXT PRIMARY KEY,
            track_name TEXT NOT NULL,
            track_genre TEXT
        )
    ''')

    df_genre = (
        df[['track_id', 'track_name', 'track_genre']]
        .drop_duplicates(subset=['track_id'])
        .fillna('')
    )

    lista_tuplas_genre = list(df_genre.itertuples(index=False, name=None))

    cursor.executemany('''
        INSERT OR REPLACE INTO genre (
            track_id, track_name, track_genre
        )
        VALUES (?, ?, ?)
    ''', lista_tuplas_genre)

    print(f"✅ Insertados {len(lista_tuplas_genre)} géneros únicos")

    # 3. Audio_Features
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Audio_Features (
            track_id TEXT PRIMARY KEY,
            danceability REAL NOT NULL,
            energy REAL NOT NULL,
            "key" INTEGER,
            loudness REAL,
            mode INTEGER,
            speechiness REAL,
            acousticness REAL,
            instrumentalness REAL,
            liveness REAL,
            valence REAL,
            tempo REAL,
            time_signature INTEGER
        )
    ''')

    df_features = (
        df[
            [
                'track_id', 'danceability', 'energy', 'key', 'loudness', 'mode',
                'speechiness', 'acousticness', 'instrumentalness', 'liveness',
                'valence', 'tempo', 'time_signature'
            ]
        ]
        .drop_duplicates(subset=['track_id'])
        .fillna(0)
    )

    lista_tuplas_features = list(df_features.itertuples(index=False, name=None))

    cursor.executemany('''
        INSERT OR REPLACE INTO Audio_Features (
            track_id, danceability, energy, "key", loudness, mode,
            speechiness, acousticness, instrumentalness, liveness,
            valence, tempo, time_signature
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', lista_tuplas_features)

    print(f"✅ Insertados {len(lista_tuplas_features)} features únicos")

    # 4. Audio_Analysis
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Audio_Analysis (
            track_id TEXT PRIMARY KEY,
            dance_energy_sum REAL NOT NULL,
            dance_energy_avg REAL NOT NULL,
            is_dance_track INTEGER,
            is_energy_high INTEGER,
            tempo_category TEXT,
            total_features_score REAL
        )
    ''')

    cursor.execute('DELETE FROM Audio_Analysis')

    cursor.execute('''
        INSERT INTO Audio_Analysis (
            track_id,
            dance_energy_sum,
            dance_energy_avg,
            is_dance_track,
            is_energy_high,
            tempo_category,
            total_features_score
        )
        SELECT
            track_id,
            (danceability + energy),
            ((danceability + energy) / 2),
            CASE WHEN (danceability + energy) > 1.2 THEN 1 ELSE 0 END,
            CASE WHEN energy > 0.7 THEN 1 ELSE 0 END,
            CASE
                WHEN tempo < 90 THEN 'Lento'
                WHEN tempo < 120 THEN 'Medio'
                ELSE 'Rapido'
            END,
            (danceability + energy + valence)
        FROM Audio_Features
    ''')

    conn.commit()
    conn.close()

    print("🎉 Base de datos creada correctamente: spotify.db")

except FileNotFoundError as e:
    print(f"❌ {e}")

except KeyError as e:
    print(f"❌ Falta una columna en el dataset: {e}")

except Exception as e:
    print(f"❌ Error general: {e}")

Dataset cargado: 114000 filas, 21 columnas
Columnas disponibles: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']
✅ Insertados 89741 álbumes únicos
✅ Insertados 89741 géneros únicos
✅ Insertados 89741 features únicos
🎉 Base de datos creada correctamente: spotify.db


In [8]:
import os
import sqlite3
import pandas as pd

# Ruta de salida
output_folder = './tablas en csv'
os.makedirs(output_folder, exist_ok=True)

# Conectar a la base de datos
conn = sqlite3.connect('spotify.db')

# Exportar tablas
df_albums = pd.read_sql_query('SELECT * FROM albums', conn)
df_genre = pd.read_sql_query('SELECT * FROM genre', conn)
df_features = pd.read_sql_query('SELECT * FROM Audio_Features', conn)

# Guardar en CSV
df_albums.to_csv(os.path.join(output_folder, 'albums.csv'), index=False, encoding='utf-8')
df_genre.to_csv(os.path.join(output_folder, 'genre.csv'), index=False, encoding='utf-8')
df_features.to_csv(os.path.join(output_folder, 'Audio_Features.csv'), index=False, encoding='utf-8')

conn.close()

print(f"✅ Exportadas 3 tablas en: {output_folder}")

✅ Exportadas 3 tablas en: ./tablas en csv


In [ ]:
import os
import sqlite3
import pandas as pd

# -------------------------
# 1) Cargar dataset original
# -------------------------
csv_path = './dataset.csv/dataset.csv'

if not os.path.isfile(csv_path):
    raise FileNotFoundError(f"No existe el archivo: {csv_path}")

df = pd.read_csv(csv_path)
print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print("Columnas disponibles:", list(df.columns))

# -------------------------
# 2) Crear / conectar BD
# -------------------------
conn = sqlite3.connect('spotify.db')
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = OFF;')

# -------------------------
# 3) Tabla albums
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS albums (
        track_id TEXT PRIMARY KEY,
        track_name TEXT NOT NULL,
        album_name TEXT,
        artists TEXT,
        popularity INTEGER
    )
''')

df_albums = (
    df[['track_id', 'track_name', 'album_name', 'artists', 'popularity']]
    .drop_duplicates(subset=['track_id'])
    .fillna('')
)
lista_tuplas_albums = list(df_albums.itertuples(index=False, name=None))

cursor.executemany('''
    INSERT OR REPLACE INTO albums (
        track_id, track_name, album_name, artists, popularity
    )
    VALUES (?, ?, ?, ?, ?)
''', lista_tuplas_albums)

print(f"✅ Insertados {len(lista_tuplas_albums)} álbumes únicos")

# -------------------------
# 4) Tabla genre
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS genre (
        track_id TEXT PRIMARY KEY,
        track_name TEXT NOT NULL,
        track_genre TEXT
    )
''')

df_genre = (
    df[['track_id', 'track_name', 'track_genre']]
    .drop_duplicates(subset=['track_id'])
    .fillna('')
)
lista_tuplas_genre = list(df_genre.itertuples(index=False, name=None))

cursor.executemany('''
    INSERT OR REPLACE INTO genre (
        track_id, track_name, track_genre
    )
    VALUES (?, ?, ?)
''', lista_tuplas_genre)

print(f"✅ Insertados {len(lista_tuplas_genre)} géneros únicos")

# -------------------------
# 5) Tabla Audio_Features
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Audio_Features (
        track_id TEXT PRIMARY KEY,
        danceability REAL NOT NULL,
        energy REAL NOT NULL,
        "key" INTEGER,
        loudness REAL,
        mode INTEGER,
        speechiness REAL,
        acousticness REAL,
        instrumentalness REAL,
        liveness REAL,
        valence REAL,
        tempo REAL,
        time_signature INTEGER
    )
''')

df_features = (
    df[
        ['track_id', 'danceability', 'energy', 'key', 'loudness', 'mode',
         'speechiness', 'acousticness', 'instrumentalness', 'liveness',
         'valence', 'tempo', 'time_signature']
    ]
    .drop_duplicates(subset=['track_id'])
    .fillna(0)
)
lista_tuplas_features = list(df_features.itertuples(index=False, name=None))

cursor.executemany('''
    INSERT OR REPLACE INTO Audio_Features (
        track_id, danceability, energy, "key", loudness, mode,
        speechiness, acousticness, instrumentalness, liveness,
        valence, tempo, time_signature
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
''', lista_tuplas_features)

print(f"✅ Insertados {len(lista_tuplas_features)} features únicos")

# -------------------------
# 6) Tabla Audio_Analysis
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Audio_Analysis (
        track_id TEXT PRIMARY KEY,
        dance_energy_sum REAL NOT NULL,
        dance_energy_avg REAL NOT NULL,
        is_dance_track INTEGER,
        is_energy_high INTEGER,
        tempo_category TEXT,
        total_features_score REAL
    )
''')

# 1. Clean up and populate the analysis table #modif
cursor.execute('DROP TABLE IF EXISTS popularity_analysis')

cursor.execute('''
    CREATE TABLE popularity_analysis AS
    SELECT 
        d.track_id,
        d.track_name,
        d.artists AS artist_id,
        g.genre_name AS track_genre,
        d.album_name AS album_id,
        d.popularity,
        af.danceability
    FROM 
        dataset d
    JOIN 
        audio_features af ON d.track_id = af.track_id
    JOIN 
        genre g ON d.genre_id = g.genre_id 
    WHERE 
        d.popularity IS NOT NULL 
        AND d.popularity > 0
    ORDER BY 
        d.popularity DESC
''')

# 2. Execute the final query to fetch the top results
cursor.execute('''
    SELECT 
        track_name, 
        track_genre, 
        popularity 
    FROM 
        popularity_analysis 
    LIMIT 15
''')

# 3. Retrieve and display the data
results = cursor.fetchall()

# Clean formatting for the output
header = f"{'Track Name':<40} | {'Genre':<20} | {'Popularity'}"
print(header)
print("-" * len(header))

for row in results:
    name, genre, pop = row
    print(f"{name:<40} | {genre:<20} | {pop}")


conn.commit()

# -------------------------
# 7) Exportar las 3 tablas a CSV
# -------------------------
output_folder = './tablas en csv'
os.makedirs(output_folder, exist_ok=True)

pd.read_sql_query('SELECT * FROM albums', conn).to_csv(
    os.path.join(output_folder, 'albums.csv'),
    index=False,
    encoding='utf-8'
)

pd.read_sql_query('SELECT * FROM genre', conn).to_csv(
    os.path.join(output_folder, 'genre.csv'),
    index=False,
    encoding='utf-8'
)

pd.read_sql_query('SELECT * FROM Audio_Analysis', conn).to_csv(
    os.path.join(output_folder, 'Audio_Analysis.csv'),
    index=False,
    encoding='utf-8'
)

conn.close()

print("🎉 Proceso completado")
print(f"📁 CSV exportados en: {output_folder}")

Dataset cargado: 114000 filas, 21 columnas
Columnas disponibles: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']
✅ Insertados 89741 álbumes únicos
✅ Insertados 89741 géneros únicos
✅ Insertados 89741 features únicos
🎉 Proceso completado
📁 CSV exportados en: ./tablas en csv


In [4]:
import sqlite3
import pandas as pd
from pathlib import Path

# -------------------------
# 1) Definir rutas del proyecto
# -------------------------
project_dir = Path(r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database")
csv_path = project_dir / "dataset.csv" / "dataset.csv"
db_path = project_dir / "spotify.db"
output_folder = project_dir / "output"

# Crear carpeta output si no existe
output_folder.mkdir(parents=True, exist_ok=True)

# Verificar existencia del CSV
if not csv_path.is_file():
    raise FileNotFoundError(f"No existe el archivo: {csv_path}")

# -------------------------
# 2) Cargar dataset original
# -------------------------
df = pd.read_csv(csv_path)
print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print("Columnas disponibles:", list(df.columns))

# -------------------------
# 3) Crear / conectar BD
# -------------------------
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = OFF;')

# -------------------------
# 4) Tabla albums
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS albums (
        track_id TEXT PRIMARY KEY,
        track_name TEXT NOT NULL,
        album_name TEXT,
        artists TEXT,
        popularity INTEGER
    )
''')

df_albums = (
    df[['track_id', 'track_name', 'album_name', 'artists', 'popularity']]
    .drop_duplicates(subset=['track_id'])
    .fillna('')
)

lista_tuplas_albums = list(df_albums.itertuples(index=False, name=None))

cursor.executemany('''
    INSERT OR REPLACE INTO albums (
        track_id, track_name, album_name, artists, popularity
    )
    VALUES (?, ?, ?, ?, ?)
''', lista_tuplas_albums)

print(f"✅ Insertados {len(lista_tuplas_albums)} álbumes únicos")

# -------------------------
# 5) Tabla genre
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS genre (
        track_id TEXT PRIMARY KEY,
        track_name TEXT NOT NULL,
        track_genre TEXT
    )
''')

df_genre = (
    df[['track_id', 'track_name', 'track_genre']]
    .drop_duplicates(subset=['track_id'])
    .fillna('')
)

lista_tuplas_genre = list(df_genre.itertuples(index=False, name=None))

cursor.executemany('''
    INSERT OR REPLACE INTO genre (
        track_id, track_name, track_genre
    )
    VALUES (?, ?, ?)
''', lista_tuplas_genre)

print(f"✅ Insertados {len(lista_tuplas_genre)} géneros únicos")

# -------------------------
# 6) Tabla Audio_Features
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Audio_Features (
        track_id TEXT PRIMARY KEY,
        danceability REAL NOT NULL,
        energy REAL NOT NULL,
        "key" INTEGER,
        loudness REAL,
        mode INTEGER,
        speechiness REAL,
        acousticness REAL,
        instrumentalness REAL,
        liveness REAL,
        valence REAL,
        tempo REAL,
        time_signature INTEGER
    )
''')

feature_cols = [
    'track_id', 'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'time_signature'
]

missing_features = [c for c in feature_cols if c not in df.columns]
if missing_features:
    raise ValueError(f"Faltan columnas en el dataset: {missing_features}")

df_features = (
    df[feature_cols]
    .drop_duplicates(subset=['track_id'])
    .fillna(0)
)

lista_tuplas_features = list(df_features.itertuples(index=False, name=None))

cursor.executemany('''
    INSERT OR REPLACE INTO Audio_Features (
        track_id, danceability, energy, "key", loudness, mode,
        speechiness, acousticness, instrumentalness, liveness,
        valence, tempo, time_signature
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
''', lista_tuplas_features)

print(f"✅ Insertados {len(lista_tuplas_features)} features únicos")

# -------------------------
# 7) Tabla Audio_Analysis
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Audio_Analysis (
        track_id TEXT PRIMARY KEY,
        dance_energy_sum REAL NOT NULL,
        dance_energy_avg REAL NOT NULL,
        is_dance_track INTEGER,
        is_energy_high INTEGER,
        tempo_category TEXT,
        total_features_score REAL
    )
''')

cursor.execute('DELETE FROM Audio_Analysis')

cursor.execute('''
    INSERT INTO Audio_Analysis (
        track_id,
        dance_energy_sum,
        dance_energy_avg,
        is_dance_track,
        is_energy_high,
        tempo_category,
        total_features_score
    )
    SELECT
        track_id,
        (danceability + energy) AS dance_energy_sum,
        ((danceability + energy) / 2) AS dance_energy_avg,
        CASE WHEN (danceability + energy) > 1.2 THEN 1 ELSE 0 END AS is_dance_track,
        CASE WHEN energy > 0.7 THEN 1 ELSE 0 END AS is_energy_high,
        CASE
            WHEN tempo < 90 THEN 'Lento'
            WHEN tempo < 120 THEN 'Medio'
            ELSE 'Rápido'
        END AS tempo_category,
        (danceability + energy + valence) AS total_features_score
    FROM Audio_Features
''')

conn.commit()

# -------------------------
# 8) Exportar tablas a CSV
# -------------------------
tablas = {
    'albums': 'albums.csv',
    'genre': 'genre.csv',
    'Audio_Features': 'Audio_Features.csv',
    'Audio_Analysis': 'Audio_Analysis.csv'
}

for tabla, archivo in tablas.items():
    df_export = pd.read_sql_query(f'SELECT * FROM {tabla}', conn)
    df_export.to_csv(
        output_folder / archivo,
        index=False,
        encoding='utf-8'
    )
    print(f"📁 Exportado: {archivo}")

# -------------------------
# 9) Vista previa rápida
# -------------------------
print("\n--- Vista previa albums ---")
print(pd.read_sql_query('SELECT * FROM albums LIMIT 5', conn))

print("\n--- Vista previa genre ---")
print(pd.read_sql_query('SELECT * FROM genre LIMIT 5', conn))

print("\n--- Vista previa Audio_Features ---")
print(pd.read_sql_query('SELECT * FROM Audio_Features LIMIT 5', conn))

print("\n--- Vista previa Audio_Analysis ---")
print(pd.read_sql_query('SELECT * FROM Audio_Analysis LIMIT 5', conn))

conn.close()

print("\n🎉 Proceso completado")
print(f"📂 Base de datos creada en: {db_path}")
print(f"📂 CSV exportados en: {output_folder}")

Dataset cargado: 114000 filas, 21 columnas
Columnas disponibles: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']
✅ Insertados 89741 álbumes únicos
✅ Insertados 89741 géneros únicos
✅ Insertados 89741 features únicos
📁 Exportado: albums.csv
📁 Exportado: genre.csv
📁 Exportado: Audio_Features.csv
📁 Exportado: Audio_Analysis.csv

--- Vista previa albums ---
                 track_id                  track_name  \
0  5SuOikwiRyPMVoIQDJUgSV                      Comedy   
1  4qPNDBW1i3p13qLCt0Ki3A            Ghost - Acoustic   
2  1iJBSr7s7jYXzM8EGcbK5b              To Begin Again   
3  6lfxq3CG4xtTiEg7opyCyx  Can't Help Falling In Love   
4  5vjLSffimiIP26QG5WcN2K                     Hold On   

                                          album_name                 arti